# 01 · Data Collection & Preparation
Goal: load the raw UCI credit-card table, check it, understand how the months line up, and save clean tables plus one fixed train/test split that every later notebook reuses.

The target we predict is the real column `default.payment.next.month` (renamed `default_next_month`). We never invent a different target.

**Setup.** Import libraries and our helper package `credit_kg` (the shared code lives in `src/credit_kg`).

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from credit_kg import config as C
from credit_kg import data as D

**Load the raw CSV.** `load_raw` also checks the 25 expected columns, unique IDs, no missing values, and a 0/1 target.

In [2]:
df = D.load_raw()
print(df.shape)
df.head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


**Data-quality report.** Counts of odd values that we must handle rather than delete: negative bills (credit balances), utilization above 100%, all-zero rows, undocumented category codes.

In [3]:
dq = D.data_quality(df)
dq.to_csv(C.TABLES / "data_quality.csv", index=False)
dq

,check,value
0,customers,30000.0000
1,columns,25.0000
2,missing_cells,0.0000
3,duplicate_ids,0.0000
4,duplicate_rows_excluding_id,35.0000
5,default_rate,0.2212
6,negative_bill_cells,3932.0000
7,rows_with_negative_bill,1930.0000
8,all_zero_bill_rows,866.0000
9,all_zero_payment_rows,1432.0000


**Class balance.** About 22% of customers default next month. Moderate imbalance, so later we judge models with recall, precision, F1 and AUC, not accuracy.

In [4]:
df["default.payment.next.month"].value_counts().rename("customers").to_frame().assign(share=lambda t: (t.customers / t.customers.sum()).round(4))

,customers,share
default.payment.next.month,,
0,23364,0.7788
1,6636,0.2212


**What do the repayment status codes look like?** Documented meaning: -2 = no consumption, -1 = paid in full, 0 = revolving (minimum paid), 1..9 = months of delay.
Watch the row for code `1`: it is common in September (PAY_0) but almost absent in the other months, so the codes are not perfectly consistent over time.

In [ ]:
PAY = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
pd.concat({c: df[c].value_counts() for c in PAY}, axis=1).sort_index().fillna(0).astype(int)

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
-2,2759,3782,4085,4348,4546,4895
-1,5686,6050,5938,5687,5539,5740
0,14737,15730,15764,16455,16947,16286
1,3688,28,4,2,0,0
2,2667,3927,3819,3159,2626,2766
3,322,326,240,180,178,184
4,76,99,76,69,84,49
5,26,25,21,35,17,13
6,11,12,23,5,4,19
7,9,20,27,58,58,46


**Are delays a clean running counter?** If code *k* really meant '*k* months late', a customer could only go up by at most 1 per month. We count how often it jumps by more. Many jumps mean we should treat the codes as ordered labels, not as exact month counts.

In [6]:
jumps = total = 0
for newer, older in zip(PAY[:-1], PAY[1:]):
    late_now = df[newer] >= 1
    total += late_now.sum()
    jumps += (df.loc[late_now, newer] > df.loc[late_now, older].clip(lower=0) + 1).sum()
print(f"late months that jump by more than +1 vs previous month: {jumps} of {total} ({jumps/total:.0%})")

late months that jump by more than +1 vs previous month: 6209 of 21947 (28%)


**Which bill does each payment pay?** We count exact matches between a payment and (a) the same month's bill and (b) the previous month's bill.
Payments match the *previous* month's bill about 6x more often, so a payment in month *t* settles the statement of month *t-1*. This decides how we compute payment coverage.

In [7]:
D.alignment_evidence(df)

,payment_column,matches_same_month_bill,matches_previous_month_bill
0,PAY_AMT1,671,4066
1,PAY_AMT2,636,4087
2,PAY_AMT3,644,4083
3,PAY_AMT4,648,4361
4,PAY_AMT5,582,4616


**Reshape to a clean wide table.** One row per customer, months renamed in time order: `m1` = April ... `m6` = September. We check that the September status really is the raw `PAY_0`.

In [8]:
wide = D.to_wide(df)
assert (wide["status_m6"] == df["PAY_0"]).all() and (wide["bill_m6"] == df["BILL_AMT1"]).all()
assert (wide["bill_m1"] == df["BILL_AMT6"]).all()
wide.head(3)

,customer_id,credit_limit,sex,education,marriage,age,status_m1,status_m2,status_m3,status_m4,status_m5,status_m6,bill_m1,bill_m2,bill_m3,bill_m4,bill_m5,bill_m6,pay_m1,pay_m2,pay_m3,pay_m4,pay_m5,pay_m6,default_next_month
0,1,20000.0,2,2,1,24,-2,-2,-1,-1,2,2,0.0,0.0,0.0,689.0,3102.0,3913.0,0.0,0.0,0.0,0.0,689.0,0.0,1
1,2,120000.0,2,2,2,26,2,0,0,0,2,-1,3261.0,3455.0,3272.0,2682.0,1725.0,2682.0,2000.0,0.0,1000.0,1000.0,1000.0,0.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,15549.0,14948.0,14331.0,13559.0,14027.0,29239.0,5000.0,1000.0,1000.0,1000.0,1500.0,1518.0,0


**Build the long monthly table.** One row per customer per month (30,000 × 6 = 180,000 rows). Each row gets a utilization, a named repayment state, a late flag, and the coverage of the previous bill by this month's payment.

In [9]:
monthly = D.to_monthly(wide)
print(monthly.shape)
monthly[monthly.customer_id == 3]

(180000, 13)


,customer_id,record_id,month_index,month,bill_amount,payment_amount,repayment_status,state,utilization,is_late,credit_balance,prev_bill_amount,coverage_of_prev_bill
12,3,3-1,1,2005-04,15549.0,5000.0,0,REVOLVING,0.172767,False,False,NaN,NaN
13,3,3-2,2,2005-05,14948.0,1000.0,0,REVOLVING,0.166089,False,False,15549.0,0.064313
14,3,3-3,3,2005-06,14331.0,1000.0,0,REVOLVING,0.159233,False,False,14948.0,0.066899
15,3,3-4,4,2005-07,13559.0,1000.0,0,REVOLVING,0.150656,False,False,14331.0,0.069779
16,3,3-5,5,2005-08,14027.0,1500.0,0,REVOLVING,0.155856,False,False,13559.0,0.110628
17,3,3-6,6,2005-09,29239.0,1518.0,0,REVOLVING,0.324878,False,False,14027.0,0.108220


**Named repayment states.** Raw codes are mapped to 5 readable states. These become shared nodes in the knowledge graph later.

In [10]:
monthly["state"].value_counts().to_frame("records")

,records
state,
REVOLVING,95919
PAID_FULL,34640
NO_USE,24415
LATE_2PLUS,21304
LATE_1,3722


**One fixed train/test split.** 80/20, stratified on the target, seed 42. Every later notebook reads this file, so every model and every population statistic uses the same customers.

In [11]:
split = D.make_split(wide)
chk = split.merge(wide[["customer_id", C.TARGET]])
chk.groupby("split")[C.TARGET].agg(customers="size", default_rate="mean").round(4)

,customers,default_rate
split,,
test,6000,0.2212
train,24000,0.2212


**Save outputs** for the next notebooks.

In [12]:
wide.to_parquet(C.PROCESSED / "customers.parquet", index=False)
monthly.to_parquet(C.PROCESSED / "monthly.parquet", index=False)
split.to_csv(C.PROCESSED / "split.csv", index=False)
print("saved:", *[p.name for p in C.PROCESSED.iterdir()])

saved: customers.parquet monthly.parquet split.csv


### Takeaways
* 30,000 customers, no missing values, target rate 22.1%.
* Payment in month *t* pays the bill of month *t-1* → coverage = `pay_t / bill_(t-1)` (5 usable pairs per customer).
* Status codes are ordered labels, not exact month counters; code `1` is rare outside September.
* Odd values (negative bills, utilization > 100%) are kept and handled explicitly, not deleted.
* `PAY_AMT` is historical **payment behaviour**, not income.